In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from scipy.special import logit

# ---------------------------------------------------------
# CONFIGURATION & STYLE
# ---------------------------------------------------------
CLINICAL_COLORS = {
    "ViennaAIdb": "#005B96",
    "MIMIC": "#00A6A6",
    "Reference": "black"
}

OUTPUT_DIR = 'model_outputs'
SAVE_PATH = 'Calibration_Combined.png'
N_BINS = 10
BIN_EDGES = np.linspace(0, 1, N_BINS + 1)
ticks = np.linspace(0, 1, 6)


# ---------------------------------------------------------
# CALIBRATION METRICS
# ---------------------------------------------------------
def calibration_metrics(y_true, y_pred, eps=1e-8):
    y_pred_clipped = np.clip(y_pred, eps, 1 - eps)
    log_odds = logit(y_pred_clipped).reshape(-1, 1)

    lr = LogisticRegression(
        penalty=None,
        solver='lbfgs',
        max_iter=10000
    )
    lr.fit(log_odds, y_true)

    intercept = lr.intercept_[0]
    slope = lr.coef_[0][0]
    return intercept, slope


# ---------------------------------------------------------
# LOADING DATA
# ---------------------------------------------------------
def load_data():
    files = {
        'ViennaAIdb': os.path.join(OUTPUT_DIR, 'calibration_internal.npz'),
        'MIMIC': os.path.join(OUTPUT_DIR, 'calibration_external.npz')
    }
    data_store = {}
    for name, filepath in files.items():
        if os.path.exists(filepath):
            data = np.load(filepath)
            data_store[name] = {
                'y_true': data['y_true'],
                'y_pred': data['y_pred_proba']
            }
            print(f"Loaded {name} data.")
        else:
            print(f"Warning: {filepath} not found.")
    return data_store


# ---------------------------------------------------------
# PLOTTING
# ---------------------------------------------------------
def plot_combined_calibration():
    data_loaded = load_data()
    if not data_loaded:
        return

    sns.set_style("whitegrid")
    sns.set_context("notebook")

    fig, (ax1, ax2) = plt.subplots(
        nrows=2, ncols=1, figsize=(8, 10),
        sharex=True, gridspec_kw={'height_ratios': [3, 1]}
    )

    # --- TOP PLOT: RELIABILITY DIAGRAM ---
    ax1.plot(
        [0, 1], [0, 1], linestyle=":",
        color=CLINICAL_COLORS["Reference"],
        label="Perfectly Calibrated", alpha=0.6
    )

    metrics_data = {}

    for name, d in data_loaded.items():
        y_true, y_pred = d['y_true'], d['y_pred']

        bs = brier_score_loss(y_true, y_pred)
        intercept, slope = calibration_metrics(y_true, y_pred)

        metrics_data[name] = {
            'brier': bs,
            'intercept': intercept,
            'slope': slope
        }

        prob_true, prob_pred = calibration_curve(
            y_true, y_pred, n_bins=N_BINS, strategy='uniform'
        )

        ax1.plot(
            prob_pred, prob_true, marker='s', linestyle='-',
            linewidth=2, markersize=6,
            color=CLINICAL_COLORS[name], label=name
        )

    # --- METRICS ANNOTATION BOX ---
    header    = f"{'':.<12s} {'Brier':>7s} {'Int.':>7s} {'Slope':>7s}"
    separator = "─" * 36
    ideal     = f"{'(ideal)':<12s} {'0':>7s} {'0':>7s} {'1':>7s}"

    metrics_rows = []
    for name, m in metrics_data.items():
        metrics_rows.append(
            f"{name:<12s} {m['brier']:>7.3f} {m['intercept']:>+7.3f} {m['slope']:>7.3f}"
        )

    box_text = "\n".join([header, separator] + metrics_rows + [separator, ideal])

    ax1.text(
        0.98, 0.04, box_text,
        transform=ax1.transAxes,
        fontsize=8.5,
        fontfamily='monospace',
        verticalalignment='bottom',
        horizontalalignment='right',
        bbox=dict(
            boxstyle='round,pad=0.5',
            facecolor='white',
            edgecolor='lightgrey',
            alpha=0.9
        )
    )

    ax1.set_ylabel("Observed Frequency")
    ax1.legend(loc="upper left", frameon=True)
    ax1.set_xlim([0, 1])
    ax1.set_ylim([0, 1])

    # --- BOTTOM PLOT: FREQUENCY HISTOGRAM ---
    combined_list = []
    for name, d in data_loaded.items():
        combined_list.append(
            pd.DataFrame({
                'Predicted Probability': d['y_pred'],
                'Dataset': name
            })
        )
    df_combined = pd.concat(combined_list)

    sns.histplot(
        data=df_combined, x='Predicted Probability', hue='Dataset',
        bins=BIN_EDGES, stat="percent", common_norm=False,
        multiple="dodge", shrink=0.8, palette=CLINICAL_COLORS,
        alpha=0.7, edgecolor="white", ax=ax2, legend=False
    )

    ax2.set_xlabel("Predicted Probability")
    ax2.set_ylabel("Percentage (%)")
    ax2.set_xticks(ticks)
    ax1.set_yticks(ticks)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.05)

    plt.savefig(SAVE_PATH, dpi=300, bbox_inches='tight')
    print(f"Plot saved to {SAVE_PATH}")
    plt.show()


if __name__ == "__main__":
    plot_combined_calibration()